In [ ]:
import json
import requests
import os
from openai import OpenAI


client = OpenAI(
    api_key="",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {   "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "Explain to me how AI works"
        }
    ]
)

print(response.choices[0].message)

def get_weather(city: str): #weather api calling logic
    url = f"https://wttr.in/{city}?format=%C+%t" #open source weather api, we don't need to create apis for weather
    response = requests.get(url)

    if response.status_code == 200:
        return f"The weather in {city} is {response.text}."
    
    return "Something went wrong"

def run_command(cmd: str): #executing system commands logic
    result = os.system(cmd)
    return result

def calculator(expression: str):
    """
    Calculator tool using eval()
    Example input:
    "10 + 20 * 5"
    """

    try:
        result = eval(expression)

        return f"The result of {expression} is {result}"

    except Exception as e:
        return f"Calculation error: {str(e)}"

def search_wikipedia(topic: str):
    """
    Wikipedia API calling logic
    """

    url = "https://en.wikipedia.org/w/api.php"

    headers = {
        "User-Agent": "MyAI-Agent/1.0"
    }

    # First search the article
    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": topic,
        "srlimit": 1,
        "format": "json"
    }

    search_response = requests.get(
        url,
        params=search_params,
        headers=headers
    )

    if search_response.status_code != 200:
        return "Wikipedia search failed"

    search_data = search_response.json()

    results = search_data["query"]["search"]

    if not results:
        return "No Wikipedia article found"


    title = results[0]["title"]


    # Get article content
    content_params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": True,
        "titles": title,
        "format": "json"
    }

    content_response = requests.get(
        url,
        params=content_params,
        headers=headers
    )

    content_data = content_response.json()

    page = next(
        iter(content_data["query"]["pages"].values())
    )

    article = page.get(
        "extract",
        "No article content found"
    )


    return f"""
Wikipedia Article: {title}

{article[:3000]}
"""

available_tools = {

    "get_weather": get_weather,

    "run_command": run_command,

    "search_wikipedia": search_wikipedia,

    "calculator": calculator
}

SYSTEM_PROMPT = f"""
    You are an helpfull AI Assistant who is specialized in resolving user query.
    You work on start, plan, action, observe mode.

    For the given user query and available tools, plan the step by step execution, based on the planning,
    select the relevant tool from the available tool. and based on the tool selection you perform an action to call the tool.
    - "search_wikipedia": Takes a topic name as input and returns information from Wikipedia.

    Wait for the observation and based on the observation from the tool call resolve the user query.

    Rules:
    - Follow the Output JSON Format.
    - Always perform one step at a time and wait for next input
    - Carefully analyse the user query
    - Don't do all steps one at a time, go one by one, step by step response

    Output JSON Format:
    {{
        "step": "string",
        "content": "string",
        "function": "The name of function if the step is action",
        "input": "The input parameter for the function",
    }}

    Available Tools:
    - "get_weather": Takes a city name as an input and returns the current weather for the city
    - "run_command": Takes linux command as a string and executes the command and returns the output after executing it.

    Example:
    User Query: What is the weather of new york?
    Output: {{ "step": "plan", "content": "The user is interseted in weather data of new york" }}
    Output: {{ "step": "plan", "content": "From the available tools I should call get_weather" }}
    Output: {{ "step": "action", "function": "get_weather", "input": "new york" }}
    Output: {{ "step": "observe", "output": "12 Degree Cel" }}
    Output: {{ "step": "output", "content": "The weather for new york seems to be 12 degrees." }}

"""

#automating the agent (llm+tools(get_weather,run_command))

messages = [
  { "role": "system", "content": SYSTEM_PROMPT }
]
while True:
    query = input("> ")
    messages.append({ "role": "user", "content": query })

    while True:
        response = client.chat.completions.create(
            model="gemini-2.5-flash",
            response_format={"type": "json_object"},
            messages=messages
        )

        messages.append({ "role": "assistant", "content": response.choices[0].message.content })
        parsed_response = json.loads(response.choices[0].message.content)

        if parsed_response.get("step") == "plan":#output will be of modes:plan,plan, action
            print(f"🧠: {parsed_response.get("content")}")
            continue

        if parsed_response.get("step") == "action":#once i hit action mode
            tool_name = parsed_response.get("function") #will get the respective function 
            tool_input = parsed_response.get("input")

            print(f"🛠️: Calling Tool:{tool_name} with input {tool_input}")

            if available_tools.get(tool_name) != False: #if that function is in available tools
                output = available_tools[tool_name](tool_input) #func() calling
                messages.append({ "role": "user", "content": json.dumps({ "step": "observe", "output": output }) }) #adding that action response in messages
                continue #go to the next mode
        
        if parsed_response.get("step") == "output":
            print(f"🤖: {parsed_response.get("content")}")
            break

ChatCompletionMessage(content='That\'s a fantastic question, and one that many people are curious about! At its core, AI isn\'t magic; it\'s a sophisticated way of getting computers to **simulate human-like intelligence**.\n\nHere\'s a breakdown of how it generally works, from simple concepts to more advanced techniques:\n\n---\n\n### The Core Idea: Learning from Data\n\nImagine you want to teach a child to recognize a cat. You wouldn\'t give them a detailed instruction manual on "cat features" (fur type, ear shape, whisker length, etc.). Instead, you\'d show them many pictures of cats, say "that\'s a cat," and many pictures of other animals, saying "that\'s not a cat." Over time, the child learns to identify a cat on their own, even if they\'ve never seen that *exact* cat before.\n\nAI works in a very similar way:\n\n1.  **Data is the Fuel:** AI systems are fed massive amounts of data – images, text, numbers, sounds, videos, etc. This data is often "labeled," meaning someone has alrea

InternalServerError: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]